Objective: Perform incremental data processing using Delta Lake.


**Step 1.** Load dataset into a Delta table.


In [0]:
df = spark.table("customer_master")

display(df)

In [0]:
print("Number of Rows :", df.count())
print("Number of Columns :", len(df.columns))

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

**Step 2)** Perform Basic Cleaning (Handle Nulls & Remove Duplicates)

In [0]:
df_clean = df.fillna("Unknown")

In [0]:
df_clean = df.fillna("Unknown")

In [0]:
df_clean = df_clean.dropDuplicates()

In [0]:
print("Original Rows :", df.count())
print("Rows After Cleaning :", df_clean.count())

In [0]:
display(df_clean)

->Delta tables do not allow spaces in column names by default. so i have renamed the column names


In [0]:
for col_name in df_clean.columns:
    df_clean = df_clean.withColumnRenamed(col_name, col_name.replace(" ", "_"))

In [0]:
df_clean.write.format("delta") .mode("overwrite") .saveAsTable("customer_master_clean")

In [0]:
print(df_clean.columns)

In [0]:
display(spark.table("customer_master_clean"))

In [0]:
master_df=spark.table("customer_master_clean")

display(master_df)

**Step 3.** Create a second dataset simulating new/incremental data.

In [0]:
incremental_df=master_df.limit(3)

In [0]:
from pyspark.sql.functions import col

incremental_df = incremental_df.withColumn(
    "Sales",
    col("Sales")+500
)

In [0]:
new_df = master_df.orderBy("Row_ID", ascending=False).limit(2)

In [0]:
from pyspark.sql.functions import when

new_df=new_df.withColumn(
    "Row_ID",
    when(col("Row_ID")==9994,9995).otherwise(9996)
)

In [0]:
new_df=new_df.withColumn("Customer_ID",
    when(col("Row_ID")==9995,"CG-99999")
    .otherwise("CG-99998")
)

In [0]:
new_df=new_df.withColumn(
    "Customer_Name",
    when(col("Row_ID")==9995,"Rohan Sharma")
    .otherwise("Ananya Verma")
)

In [0]:
customer_incremental=incremental_df.union(new_df)
display(customer_incremental)

In [0]:
display(customer_incremental)

In [0]:
%sql
SHOW TABLES;

In [0]:
customer_incremental.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("customer_incremental")

In [0]:
display(spark.table("customer_incremental"))

In [0]:
spark.table("customer_master_clean").createOrReplaceTempView("customer_master")
spark.table("customer_incremental").createOrReplaceTempView("customer_incremental")

**step 4.** Apply MERGE operation to update existing and insert new records.

In [0]:
customer_incremental = customer_incremental.dropDuplicates(["Row_ID"])

In [0]:
customer_incremental.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_incremental")

In [0]:
%sql
MERGE INTO customer_master AS target
USING customer_incremental AS source
ON target.Row_ID = source.Row_ID
WHEN MATCHED THEN
UPDATE SET
target.Customer_Name = source.Customer_Name,
target.Sales = source.Sales,
target.Profit = source.Profit
WHEN NOT MATCHED THEN
INSERT *


In [0]:
final_df = spark.table("customer_master_clean")
display(final_df)

**Step 5.** Validate results (row count, duplicates).

In [0]:
print("Total Rows :", final_df.count())

In [0]:
from pyspark.sql.functions import count

final_df.groupBy("Row_ID") \
.agg(count("*").alias("count")) \
.filter("count>1") \
.show()

In [0]:
display(final_df.limit(10))

**Step 6.** Display final dataset and summary.

In [0]:
display(customer_incremental)

# SUMMARY

This assignment focused on understanding and implementing **incremental data processing using Delta Lake** in Databricks. The objective was to load an existing dataset into a Delta table, perform data cleaning, simulate incremental data, apply the Delta Lake MERGE operation, and validate the final output.

I started by loading the **customer_master** dataset into a Spark DataFrame using the existing Delta table and displayed the dataset to understand its structure. The dataset contained **9,994 records** and **21 columns**. I verified the schema using `printSchema()` and previewed the first few records to understand the available attributes such as Customer ID, Customer Name, Product details, Sales, Quantity, Discount, and Profit.

The next step was data cleaning. I handled missing values by replacing all null values with **"Unknown"** using the `fillna()` function. Although the dataset did not contain significant missing values, this step ensured consistency in the data. I then removed duplicate records using `dropDuplicates()` and compared the row count before and after cleaning.

* **Original number of records:** 9,994
* **Number of records after cleaning:** 9,994

Since both counts remained the same, it confirmed that the dataset did not contain duplicate records.

While converting the cleaned data into a Delta table, I encountered an issue because Delta Lake does not allow spaces in column names by default. To resolve this, I renamed all columns by replacing spaces with underscores using `withColumnRenamed()`. For example:

* `Row ID` → `Row_ID`
* `Order ID` → `Order_ID`
* `Order Date` → `Order_Date`
* `Ship Date` → `Ship_Date`
* `Customer ID` → `Customer_ID`
* `Customer Name` → `Customer_Name`
* `Postal Code` → `Postal_Code`
* `Product ID` → `Product_ID`
* `Product Name` → `Product_Name`
* `Sub-Category` → `Sub_Category`

This made the schema compatible with Delta Lake. The cleaned dataset was then saved as a Delta table named **customer_master_clean**.

To simulate incremental processing, I created a second dataset named **customer_incremental**. I selected the first three records from the master dataset and modified their **Sales** values by increasing them, representing updated transactions for existing customers. I also selected the last two records from the master dataset and modified their **Row_ID**, **Customer_ID**, and **Customer_Name** to simulate two completely new customer records. These updated and new records were combined using the `union()` operation to form the incremental dataset.

The incremental dataset was saved as another Delta table named **customer_incremental** and verified by displaying its contents.

For incremental processing, I created temporary views for both Delta tables and applied the **MERGE** operation. During the merge:

* Existing records were identified using the **Row_ID** column.
* Matching records were updated by modifying the **Customer_Name**, **Sales**, and **Profit** columns.
* Records that did not exist in the master table were inserted as new records using the `WHEN NOT MATCHED THEN INSERT` clause.

This demonstrated the Delta Lake capability of performing update and insert operations within a single transaction.

After completing the MERGE operation, I validated the final output by loading the updated Delta table. I verified the total number of records and checked for duplicate **Row_ID** values using aggregation. No duplicate Row_ID values were found, confirming that the merge operation maintained data integrity.

Finally, I displayed the updated Delta table to verify that both updated and newly inserted customer records were present. This confirmed that the incremental processing workflow had executed successfully.

Overall, this assignment helped me understand the practical implementation of Delta Lake for incremental data processing. I learned how to clean data, resolve schema compatibility issues, create Delta tables, simulate incremental data, perform MERGE operations, and validate the final dataset. The assignment also provided hands-on experience with PySpark DataFrames, Delta Lake storage format, and SQL-based MERGE operations in Databricks, giving me a clear understanding of how incremental data pipelines are implemented in real-world data engineering workflows.
